## Bronze Layer
Raw extract of the three source systems, as landed.

In [1]:
import pandas as pd

In [2]:
shipments = pd.read_csv("sample_data/raw/shipment_master.csv")
shipments_df = pd.DataFrame(shipments)
shipments_df.head()

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg
0,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62
1,SHP-00000001,2011-07-13T17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23
2,SHP-00000002,2011-07-13T16:58:01.421036,2011-07-16,GTA,COMMERCIAL,GROUND,False,18.53
3,SHP-00000003,2011-07-13T07:07:17.617848,2011-07-16,GTA,RESIDENTIAL,GROUND,False,24.06
4,SHP-00000004,2011-07-13T09:43:06.971432,2011-07-13,Montreal,RESIDENTIAL,SAME_DAY,False,1.71


In [3]:
print(shipments_df.shape)

(1066714, 8)


In [4]:
events = pd.read_json("sample_data/raw/raw_scan_events.jsonl", lines=True)
events_df = pd.DataFrame(events)
events_df.head()

,event_id,shipment_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result
0,08ceb455-a952-46f0-9678-ab41080537c2,SHP-00000000,PICKUP,2011-07-13T11:46:59.481209,ATL-HUB-01,2011-07-13,NaN,NaN
1,b359356b-6387-4e9c-9db0-1c29d5bba769,SHP-00000000,HUB_SCAN,2011-07-13T18:01:17.708533,ATL-HUB-01,2011-07-13,NaN,NaN
2,373aae99-cf47-470b-b7c5-e8172ebbb2ad,SHP-00000000,DELIVERY_ATTEMPT,2011-07-14T21:30:40.697226,ATL-HUB-01,2011-07-14,1.0,SUCCESS
3,62bd5cb0-c9b7-46ec-9896-97078fc63f13,SHP-00000001,PICKUP,2011-07-13T21:09:51.206677,CAL-HUB-01,2011-07-13,NaN,NaN
4,81c1c42e-4776-48f3-afb0-23a189635f1b,SHP-00000001,HUB_SCAN,2011-07-14T02:58:17.422049,CAL-HUB-01,2011-07-14,NaN,NaN


In [5]:
print(events_df.shape)

(3345616, 8)


In [6]:
billing = pd.read_csv("sample_data/raw/billing_extract.csv")
billing_df = pd.DataFrame(billing)
billing_df.head()

,shipment_id,invoice_date,base_rate,fuel_surcharge_pct,fuel_surcharge_amount,guarantee_fee,total_billed
0,SHP-00000000,2011-07-14,8.00,0.1478,1.18,0.0,9.18
1,SHP-00000001,2011-07-14,27.88,0.1478,4.12,0.0,32.00
2,SHP-00000002,2011-07-14,22.24,0.1478,3.29,0.0,25.53
3,SHP-00000003,2011-07-13,28.87,0.1478,4.27,0.0,33.14
4,SHP-00000004,2011-07-13,25.00,0.1478,3.69,0.0,28.69


In [7]:
print(billing.shape)

(1066714, 7)


## Silver Layer
Cleaned and validated data: quarantine of bad records (orphaned scan events, duplicate SUCCESS deliveries), FDA derivation, breach detection, billing join, and credit liability calculation.

In [8]:
# orphaned rows and duplicate shipment_id rows
orphaned_billing = billing_df[~billing_df['shipment_id'].isin(shipments_df['shipment_id'])]
duplicate_billing = billing_df[billing_df.duplicated('shipment_id', keep=False)]
print(f"orphaned billing rows: {len(orphaned_billing)}")
print(f"duplicate shipment_id billing rows: {len(duplicate_billing)}")

orphaned billing rows: 0
duplicate shipment_id billing rows: 0


In [9]:
# quarantine both - save for investigation, then exclude from billing_df
pd.concat([orphaned_billing, duplicate_billing]).drop_duplicates().to_csv("sample_data/output/quarantined_billing.csv", index=False)

billing_df = billing_df[
    billing_df['shipment_id'].isin(shipments_df['shipment_id']) &
    ~billing_df.duplicated('shipment_id', keep=False)
]

In [10]:
# checks for shipments that have zero scan events (orphaned shipments)
df = pd.merge(shipments_df, events_df, how='left', left_on='shipment_id', right_on='shipment_id')
df.head()

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg,event_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result
0,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,08ceb455-a952-46f0-9678-ab41080537c2,PICKUP,2011-07-13T11:46:59.481209,ATL-HUB-01,2011-07-13,NaN,NaN
1,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,b359356b-6387-4e9c-9db0-1c29d5bba769,HUB_SCAN,2011-07-13T18:01:17.708533,ATL-HUB-01,2011-07-13,NaN,NaN
2,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,373aae99-cf47-470b-b7c5-e8172ebbb2ad,DELIVERY_ATTEMPT,2011-07-14T21:30:40.697226,ATL-HUB-01,2011-07-14,1.0,SUCCESS
3,SHP-00000001,2011-07-13T17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23,62bd5cb0-c9b7-46ec-9896-97078fc63f13,PICKUP,2011-07-13T21:09:51.206677,CAL-HUB-01,2011-07-13,NaN,NaN
4,SHP-00000001,2011-07-13T17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23,81c1c42e-4776-48f3-afb0-23a189635f1b,HUB_SCAN,2011-07-14T02:58:17.422049,CAL-HUB-01,2011-07-14,NaN,NaN


In [11]:
df[df['event_id'].isnull()]

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg,event_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result


In [12]:
# events that have a shipment_id that does not exist in the shipments df (aka orphaned events: scanned by mistake, shipment was deleted, etc.)
orphaned = events_df[~events_df['shipment_id'].isin(shipments_df['shipment_id'])]
orphaned

,event_id,shipment_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result
3340283,1fdc7de7-01e4-434b-baf9-9c32905a90c7,SHP-ORPHAN-957012,HUB_SCAN,2018-05-06T08:22:09.726484,VAN-HUB-02,2018-05-06,NaN,NaN
3340284,345bdfc1-7e8b-48b8-8fb0-c5a232b8ad00,SHP-ORPHAN-277969,HUB_SCAN,2015-07-31T02:31:59.248961,EDM-HUB-01,2015-07-31,NaN,NaN
3340285,80fc3118-6557-48ab-b9df-be0ced38ce28,SHP-ORPHAN-889330,HUB_SCAN,2019-01-10T13:40:39.096862,VAN-HUB-01,2019-01-10,NaN,NaN
3340286,e438b824-ba41-4c58-851f-e6018647d309,SHP-ORPHAN-629753,HUB_SCAN,2012-06-30T20:38:07.414986,CAL-HUB-01,2012-06-30,NaN,NaN
3340287,bbfb28cd-9881-45ff-b583-b644790e629b,SHP-ORPHAN-585226,HUB_SCAN,2022-10-27T09:59:16.660126,ATL-HUB-01,2022-10-27,NaN,NaN
...,...,...,...,...,...,...,...,...
3345611,ac619744-751a-4b18-bd74-58e4f9824656,SHP-ORPHAN-733086,HUB_SCAN,2026-03-16T15:32:28.053838,MTL-HUB-02,2026-03-16,NaN,NaN
3345612,2f188e41-de15-442f-8e4e-e9333ace49d8,SHP-ORPHAN-402361,HUB_SCAN,2013-07-12T04:25:01.466151,VAN-HUB-01,2013-07-12,NaN,NaN
3345613,dfbd8251-2c30-4f73-96e3-893e40419095,SHP-ORPHAN-348840,HUB_SCAN,2017-05-05T13:47:10.066117,MTL-HUB-02,2017-05-05,NaN,NaN
3345614,159145c5-c986-42d7-bb51-dd184296a6fd,SHP-ORPHAN-973676,HUB_SCAN,2012-01-10T05:09:36.876967,EDM-HUB-01,2012-01-10,NaN,NaN


In [13]:
orphaned.to_csv("sample_data/output/orphaned_events.csv", index=False)

In [14]:
# cleans the events df so it will only contain events that have a shipment_id that exists in the shipments df
events_df = events_df[events_df['shipment_id'].isin(shipments_df['shipment_id'])]

In [15]:
print(events_df.shape)

(3340283, 8)


In [16]:
# checks for duplicate event_id rows in events_df (exact-duplicate ingestion)
duplicate_events = events_df[events_df.duplicated('event_id', keep=False)]
print(f"duplicate event_id rows: {len(duplicate_events)}")

duplicate event_id rows: 0


In [17]:
# quarantines duplicate event_id rows, flags true conflicts (differ beyond event_id) vs exact copies, then dedupes
duplicate_events.to_csv("sample_data/output/duplicate_events.csv", index=False)

conflicting_duplicate_events = duplicate_events[~duplicate_events.duplicated(keep=False)]
if len(conflicting_duplicate_events) > 0:
    print(f"WARNING: {len(conflicting_duplicate_events)} duplicate event_id rows conflict beyond event_id (not exact copies) - keeping 'first' arbitrarily")
else:
    print("all duplicate event_id rows are exact copies")

events_df = events_df.drop_duplicates('event_id', keep='first')

all duplicate event_id rows are exact copies


In [18]:
# derives the actual attempt_result domain observed in the data (do not assume it's just SUCCESS/FAILURE)
events_df[events_df['event_type'] == 'DELIVERY_ATTEMPT']['attempt_result'].value_counts(dropna=False)

attempt_result
SUCCESS               1069755
FAILED_NO_ONE_HOME     148653
FAILED_NO_ACCESS        51168
FAILED_REFUSED          22345
Name: count, dtype: int64

In [19]:
# flags DELIVERY_ATTEMPT rows with an attempt_result outside the observed valid domain (or missing) - list confirmed from the value_counts above
valid_attempt_results = ["SUCCESS", "FAILED_NO_ONE_HOME", "FAILED_NO_ACCESS", "FAILED_REFUSED"]
invalid_attempts = events_df[
    (events_df['event_type'] == 'DELIVERY_ATTEMPT') &
    (events_df['attempt_result'].isna() | ~events_df['attempt_result'].isin(valid_attempt_results))
]
print(f"invalid attempt_result rows: {len(invalid_attempts)}")
invalid_attempts.to_csv("sample_data/output/invalid_attempt_results.csv", index=False)

invalid attempt_result rows: 0


In [20]:
# checks for shipments with zero scan events - not filtered out of shipments_df/final_df, just made traceable
zero_scan_shipments = df[df['event_id'].isnull()][['shipment_id']].drop_duplicates()
print(f"zero-scan-event shipments: {len(zero_scan_shipments)}")
zero_scan_shipments.to_csv("sample_data/output/zero_scan_shipments.csv", index=False)

zero-scan-event shipments: 0


In [21]:
# checks for scan events with implausible timestamps: in the future, or before the shipment's own booking_timestamp
# (df already carries both event_timestamp and booking_timestamp from the earlier shipments/events merge)
# format='ISO8601' handles the mix of timestamps with/without a microseconds component
event_ts = pd.to_datetime(df['event_timestamp'], format='ISO8601')
booking_ts = pd.to_datetime(df['booking_timestamp'], format='ISO8601')

# "future" is relative to the as-of boundary the generator actually respected (max booking_timestamp),
# not real wall-clock time - comparing to pd.Timestamp.now() would drift as more time passes between generation and analysis
as_of = pd.to_datetime(shipments_df['booking_timestamp'], format='ISO8601').max()
future_events = events_df[pd.to_datetime(events_df['event_timestamp'], format='ISO8601') > as_of]
pre_booking_events = df[event_ts < booking_ts]

print(f"as_of boundary (max booking_timestamp): {as_of}")
print(f"future-dated events: {len(future_events)}")
print(f"pre-booking events: {len(pre_booking_events)}")

as_of boundary (max booking_timestamp): 2026-07-13 19:58:45.136195
future-dated events: 922
pre-booking events: 0


In [22]:
# combines both timestamp anomalies into a single artifact for review - not auto-filtered, since forcibly dropping them could mask a real upstream bug
timestamp_anomalies = pd.concat([future_events, pre_booking_events]).drop_duplicates()
print(f"combined timestamp anomalies (deduped): {len(timestamp_anomalies)}")
timestamp_anomalies.to_csv("sample_data/output/timestamp_anomalies.csv", index=False)

combined timestamp anomalies (deduped): 922


In [23]:
# shipments with more than one successful delivery attempt (more than one successful event_type = DELIVERY ATTEMPT event)
bad_shipments = events_df[events_df['attempt_result'] == 'SUCCESS'].groupby('shipment_id').agg({'event_id': 'count'}).reset_index().rename(columns={'event_id': 'successful_scan_count'}).query('successful_scan_count > 1')
bad_shipments

,shipment_id,successful_scan_count
1427,SHP-00001427,2
1517,SHP-00001517,2
1859,SHP-00001859,2
1937,SHP-00001937,2
2011,SHP-00002011,2
...,...,...
1065271,SHP-01065271,2
1065314,SHP-01065314,2
1066071,SHP-01066218,2
1066117,SHP-01066277,2


In [24]:
# cleans events and merged df so they will not contain shipments that have more than one successful delivery attempt
events_df = events_df[~events_df['shipment_id'].isin(bad_shipments['shipment_id'])]
df = df[~df['shipment_id'].isin(bad_shipments['shipment_id'])]

In [25]:
double_delivered_shipments = events_df[events_df['shipment_id'].isin(bad_shipments['shipment_id'])]
double_delivered_shipments.to_csv("sample_data/output/double_delivered_shipments.csv", index=False)

In [26]:
# shows the attempts for each shipment and the sequence of the attempts (0, 1, 2, etc.)
attempts = events_df[events_df['event_type']=='DELIVERY_ATTEMPT'].sort_values(['shipment_id', 'event_timestamp', 'event_id'])
attempts['attempt_seq'] = attempts.groupby('shipment_id').cumcount()
attempts.head(10)

,event_id,shipment_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result,attempt_seq
2,373aae99-cf47-470b-b7c5-e8172ebbb2ad,SHP-00000000,DELIVERY_ATTEMPT,2011-07-14T21:30:40.697226,ATL-HUB-01,2011-07-14,1.0,SUCCESS,0
5,e151a653-7bb8-4e9a-8504-5d71d374f84a,SHP-00000001,DELIVERY_ATTEMPT,2011-07-14T14:08:39.170847,CAL-HUB-01,2011-07-14,1.0,SUCCESS,0
8,39587c20-eae9-4ce8-9897-02a168044e60,SHP-00000002,DELIVERY_ATTEMPT,2011-07-14T11:48:00.074634,GTA-HUB-02,2011-07-14,1.0,SUCCESS,0
11,81b77f81-1984-41bf-8c31-c1840c40759f,SHP-00000003,DELIVERY_ATTEMPT,2011-07-14T06:13:01.183364,GTA-HUB-03,2011-07-14,1.0,FAILED_NO_ONE_HOME,0
12,4402961f-4efc-48b7-9911-98ae47ee295c,SHP-00000003,DELIVERY_ATTEMPT,2011-07-15T13:21:04.410797,GTA-HUB-03,2011-07-15,2.0,SUCCESS,1
14,b27ccdc2-1de5-4fcc-8b14-6767bbd52699,SHP-00000004,DELIVERY_ATTEMPT,2011-07-13T16:37:13.848101,MTL-HUB-01,2011-07-13,1.0,FAILED_NO_ONE_HOME,0
15,a04d7c96-9be1-4a2b-8e6d-ccfdb6b65ba8,SHP-00000004,DELIVERY_ATTEMPT,2011-07-15T00:04:27.927938,MTL-HUB-01,2011-07-15,2.0,SUCCESS,1
18,77c11eb3-b0f1-453c-8706-b902a0570d80,SHP-00000005,DELIVERY_ATTEMPT,2011-07-15T06:25:15.687931,MTL-HUB-02,2011-07-15,1.0,SUCCESS,0
21,ea991bb7-9e34-4002-955f-4e7ad3c96d7e,SHP-00000006,DELIVERY_ATTEMPT,2011-07-15T00:48:27.777943,CAL-HUB-01,2011-07-15,1.0,FAILED_NO_ONE_HOME,0
22,04bf4b5d-a92d-4c9c-b49c-eadb591392de,SHP-00000006,DELIVERY_ATTEMPT,2011-07-16T03:49:16.216032,CAL-HUB-01,2011-07-16,2.0,FAILED_NO_ONE_HOME,1


In [27]:
print(attempts.shape)

(1284708, 9)


In [28]:
# calculates the first delivery attempt success (fda) boolean for each shipment
first_attempts = attempts[attempts['attempt_seq'] == 0].copy()
# an invalid/missing attempt_result must not silently resolve to fda=0 - carry a data-quality flag and leave fda unset instead
first_attempts['fda_data_quality_flag'] = (
    first_attempts['attempt_result'].isna() | ~first_attempts['attempt_result'].isin(valid_attempt_results)
)
first_attempts['fda'] = (first_attempts['attempt_result'] == 'SUCCESS').astype('Int64')
first_attempts.loc[first_attempts['fda_data_quality_flag'], 'fda'] = pd.NA
first_attempts['first_attempt_date'] = pd.to_datetime(first_attempts['event_timestamp']).dt.date
first_attempts.head(10)

,event_id,shipment_id,event_type,event_timestamp,carrier_hub,ingestion_date,attempt_number,attempt_result,attempt_seq,fda_data_quality_flag,fda,first_attempt_date
2,373aae99-cf47-470b-b7c5-e8172ebbb2ad,SHP-00000000,DELIVERY_ATTEMPT,2011-07-14T21:30:40.697226,ATL-HUB-01,2011-07-14,1.0,SUCCESS,0,False,1,2011-07-14
5,e151a653-7bb8-4e9a-8504-5d71d374f84a,SHP-00000001,DELIVERY_ATTEMPT,2011-07-14T14:08:39.170847,CAL-HUB-01,2011-07-14,1.0,SUCCESS,0,False,1,2011-07-14
8,39587c20-eae9-4ce8-9897-02a168044e60,SHP-00000002,DELIVERY_ATTEMPT,2011-07-14T11:48:00.074634,GTA-HUB-02,2011-07-14,1.0,SUCCESS,0,False,1,2011-07-14
11,81b77f81-1984-41bf-8c31-c1840c40759f,SHP-00000003,DELIVERY_ATTEMPT,2011-07-14T06:13:01.183364,GTA-HUB-03,2011-07-14,1.0,FAILED_NO_ONE_HOME,0,False,0,2011-07-14
14,b27ccdc2-1de5-4fcc-8b14-6767bbd52699,SHP-00000004,DELIVERY_ATTEMPT,2011-07-13T16:37:13.848101,MTL-HUB-01,2011-07-13,1.0,FAILED_NO_ONE_HOME,0,False,0,2011-07-13
18,77c11eb3-b0f1-453c-8706-b902a0570d80,SHP-00000005,DELIVERY_ATTEMPT,2011-07-15T06:25:15.687931,MTL-HUB-02,2011-07-15,1.0,SUCCESS,0,False,1,2011-07-15
21,ea991bb7-9e34-4002-955f-4e7ad3c96d7e,SHP-00000006,DELIVERY_ATTEMPT,2011-07-15T00:48:27.777943,CAL-HUB-01,2011-07-15,1.0,FAILED_NO_ONE_HOME,0,False,0,2011-07-15
26,2b22370f-2e53-471e-9975-f0693997eb97,SHP-00000007,DELIVERY_ATTEMPT,2011-07-14T20:22:04.342673,GTA-HUB-02,2011-07-14,1.0,SUCCESS,0,False,1,2011-07-14
29,8b73789e-03fc-437f-aa0a-13b52e49a1cb,SHP-00000008,DELIVERY_ATTEMPT,2011-07-15T02:52:08.021181,MTL-HUB-02,2011-07-15,1.0,FAILED_NO_ONE_HOME,0,False,0,2011-07-15
33,254fc61b-0aab-4918-babc-479991129300,SHP-00000009,DELIVERY_ATTEMPT,2011-07-14T17:50:46.824467,EDM-HUB-01,2011-07-14,1.0,SUCCESS,0,False,1,2011-07-14


In [29]:
actual_delivery = attempts[attempts['attempt_result'] == 'SUCCESS'][['shipment_id', 'event_timestamp']].rename(columns={'event_timestamp': 'actual_delivery_date'})
actual_delivery['actual_delivery_date'] = pd.to_datetime(actual_delivery['actual_delivery_date']).dt.date
actual_delivery.head()

,shipment_id,actual_delivery_date
2,SHP-00000000,2011-07-14
5,SHP-00000001,2011-07-14
8,SHP-00000002,2011-07-14
12,SHP-00000003,2011-07-15
15,SHP-00000004,2011-07-15


In [30]:
print(actual_delivery.shape)

(1063205, 2)


In [31]:
# merges all the relevant data into a single df and calculates whether the shipment was delivered after the promised delivery date (is_breached)
final_df = shipments_df.merge(first_attempts[['shipment_id', 'fda', 'first_attempt_date', 'fda_data_quality_flag']], how='left', on='shipment_id').merge(actual_delivery, how='left', on='shipment_id').merge(billing_df, how='left', on='shipment_id')
final_df['promised_delivery_date'] = pd.to_datetime(final_df['promised_delivery_date']).dt.date
final_df['actual_delivery_date'] = pd.to_datetime(final_df['actual_delivery_date']).dt.date
final_df['is_breached'] = (final_df['actual_delivery_date'] > final_df['promised_delivery_date']).astype('boolean')
final_df.loc[final_df['actual_delivery_date'].isna(), 'is_breached'] = pd.NA
final_df

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg,fda,first_attempt_date,fda_data_quality_flag,actual_delivery_date,invoice_date,base_rate,fuel_surcharge_pct,fuel_surcharge_amount,guarantee_fee,total_billed,is_breached
0,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,1,2011-07-14,False,2011-07-14,2011-07-14,8.00,0.1478,1.18,0.0,9.18,False
1,SHP-00000001,2011-07-13T17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23,1,2011-07-14,False,2011-07-14,2011-07-14,27.88,0.1478,4.12,0.0,32.00,False
2,SHP-00000002,2011-07-13T16:58:01.421036,2011-07-16,GTA,COMMERCIAL,GROUND,False,18.53,1,2011-07-14,False,2011-07-14,2011-07-14,22.24,0.1478,3.29,0.0,25.53,False
3,SHP-00000003,2011-07-13T07:07:17.617848,2011-07-16,GTA,RESIDENTIAL,GROUND,False,24.06,0,2011-07-14,False,2011-07-15,2011-07-13,28.87,0.1478,4.27,0.0,33.14,False
4,SHP-00000004,2011-07-13T09:43:06.971432,2011-07-13,Montreal,RESIDENTIAL,SAME_DAY,False,1.71,0,2011-07-13,False,2011-07-15,2011-07-13,25.00,0.1478,3.69,0.0,28.69,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066709,SHP-01066709,2026-07-13T12:19:53.722430,2026-07-16,GTA,RESIDENTIAL,GROUND,False,24.01,1,2026-07-14,False,2026-07-14,2026-07-14,44.90,0.0882,3.96,0.0,48.86,False
1066710,SHP-01066710,2026-07-13T09:43:05.759047,2026-07-16,Calgary-Edmonton,COMMERCIAL,GROUND,False,1.02,1,2026-07-14,False,2026-07-14,2026-07-13,12.47,0.0882,1.10,0.0,13.57,False
1066711,SHP-01066711,2026-07-13T08:52:48.048674,2026-07-13,GTA,RESIDENTIAL,SAME_DAY,False,4.09,1,2026-07-13,False,2026-07-13,2026-07-13,38.96,0.0882,3.44,0.0,42.40,False
1066712,SHP-01066712,2026-07-13T07:09:08.626625,2026-07-16,GTA,COMMERCIAL,GROUND,False,14.33,0,2026-07-14,False,NaT,2026-07-13,26.80,0.0882,2.36,0.0,29.16,<NA>


In [32]:
final_df['credit_liability'] = 0.0
breach_mask = final_df['is_guaranteed'] & final_df['is_breached'].fillna(False)
final_df.loc[breach_mask, 'credit_liability'] = final_df['total_billed'] * 0.5
final_df

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg,fda,first_attempt_date,fda_data_quality_flag,actual_delivery_date,invoice_date,base_rate,fuel_surcharge_pct,fuel_surcharge_amount,guarantee_fee,total_billed,is_breached,credit_liability
0,SHP-00000000,2011-07-13T08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,1,2011-07-14,False,2011-07-14,2011-07-14,8.00,0.1478,1.18,0.0,9.18,False,0.0
1,SHP-00000001,2011-07-13T17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23,1,2011-07-14,False,2011-07-14,2011-07-14,27.88,0.1478,4.12,0.0,32.00,False,0.0
2,SHP-00000002,2011-07-13T16:58:01.421036,2011-07-16,GTA,COMMERCIAL,GROUND,False,18.53,1,2011-07-14,False,2011-07-14,2011-07-14,22.24,0.1478,3.29,0.0,25.53,False,0.0
3,SHP-00000003,2011-07-13T07:07:17.617848,2011-07-16,GTA,RESIDENTIAL,GROUND,False,24.06,0,2011-07-14,False,2011-07-15,2011-07-13,28.87,0.1478,4.27,0.0,33.14,False,0.0
4,SHP-00000004,2011-07-13T09:43:06.971432,2011-07-13,Montreal,RESIDENTIAL,SAME_DAY,False,1.71,0,2011-07-13,False,2011-07-15,2011-07-13,25.00,0.1478,3.69,0.0,28.69,True,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066709,SHP-01066709,2026-07-13T12:19:53.722430,2026-07-16,GTA,RESIDENTIAL,GROUND,False,24.01,1,2026-07-14,False,2026-07-14,2026-07-14,44.90,0.0882,3.96,0.0,48.86,False,0.0
1066710,SHP-01066710,2026-07-13T09:43:05.759047,2026-07-16,Calgary-Edmonton,COMMERCIAL,GROUND,False,1.02,1,2026-07-14,False,2026-07-14,2026-07-13,12.47,0.0882,1.10,0.0,13.57,False,0.0
1066711,SHP-01066711,2026-07-13T08:52:48.048674,2026-07-13,GTA,RESIDENTIAL,SAME_DAY,False,4.09,1,2026-07-13,False,2026-07-13,2026-07-13,38.96,0.0882,3.44,0.0,42.40,False,0.0
1066712,SHP-01066712,2026-07-13T07:09:08.626625,2026-07-16,GTA,COMMERCIAL,GROUND,False,14.33,0,2026-07-14,False,NaT,2026-07-13,26.80,0.0882,2.36,0.0,29.16,<NA>,0.0


In [33]:
# saves the final dataset for gold layer
final_df.to_csv("sample_data/output/final_shipments.csv", index=False)